# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step workflow for loading and exploring the FAIR² "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL (Croissant schema)
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is a rich object, not a dict

print(f"Dataset: {metadata.name if hasattr(metadata, 'name') else ''}")
print(f"Description: {metadata.description if hasattr(metadata, 'description') else ''}")

## 2. Data Overview
Review the available record sets, fields, and their `@id` fields.

Below, we enumerate the record sets and their key fields to help with dataset navigation. All references are made using the entity `@id`.

In [ ]:
# List all record sets and their fields by @id
print("Available record sets and their fields:")
record_set_ids = []

for rs in dataset.record_sets:
    print(f"- RecordSet @id: {rs.id}")
    record_set_ids.append(rs.id)
    fields = getattr(rs, 'fields', [])
    for field in fields:
        print(f"    - Field @id: {field.id} (type: {getattr(field, 'data_type', 'unknown')}, name: {getattr(field, 'name', field.id)})")
if not record_set_ids:
    print("(No record sets found, or Croissant definition is referencing them indirectly. Explicit inspection follows below.)")

## 2.1. Preview Example Records
Review a sample of records from one of the record sets. Update the `record_set_id` variable below to correspond to the `@id` listed above (if the dataset contains multiple record sets).

In [ ]:
# Try to preview the first few records from the first record set
if record_set_ids:
    preview_record_set = record_set_ids[0]
    print(f"\nSample records for RecordSet @id: '{preview_record_set}':")
    for i, rec in enumerate(dataset.records(record_set=preview_record_set)):
        print(rec)
        if i >= 2:
            print("...")
            break
else:
    print("No explicit record set found to preview samples.")

## 3. Data Extraction
Load data from each record set (`@id`) into a pandas DataFrame for analysis.

Below, all available record sets from the source dataset are loaded into the workspace as DataFrames for further manipulation.

In [ ]:
# Load records from all found record sets into DataFrames
dataframes = dict()

for rec_set_id in record_set_ids:
    print(f"Loading records for RecordSet @id: {rec_set_id}")
    records = list(dataset.records(record_set=rec_set_id))
    if records:
        dataframes[rec_set_id] = pd.DataFrame(records)
        print(f"  Loaded {len(records)} records with columns: {dataframes[rec_set_id].columns.tolist()}")
    else:
        print(f"  No records found for RecordSet @id: {rec_set_id}")
if not dataframes:
    print("No tabular record sets with loadable data detected.")

# Preview the first DataFrame
if dataframes:
    primary_rsid = list(dataframes.keys())[0]
    print(f"\nPreviewing columns for RecordSet @id: {primary_rsid}")
    print(dataframes[primary_rsid].columns.tolist())
    dataframes[primary_rsid].head()
else:
    print("No DataFrames to preview.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering, normalization, and grouping by field. Use field `@id` as the key for all manipulations.

In [ ]:
# Example EDA: select a numeric field, filter, normalize, group
# You may need to adjust the field IDs based on dataset exploration in section 2.

if dataframes:
    df = dataframes[primary_rsid]

    # Try to auto-detect a likely numeric field (e.g., abundance, coverage, MW)
    import numpy as np

    # Fallback if none detected automatically
    candidate_numeric = None
    for col in df.columns:
        # Heuristic: check for float/int dtypes or numeric string columns
        if pd.api.types.is_numeric_dtype(df[col]):
            candidate_numeric = col
            break
        # Sometimes all numeric columns are parsed as string; try to cast
        try:
            maybe_numeric = pd.to_numeric(df[col], errors='coerce')
            if maybe_numeric.notna().sum() > 0:
                candidate_numeric = col
                df[col] = maybe_numeric
                break
        except Exception:
            continue

    if candidate_numeric:
        numeric_field_id = candidate_numeric
        print(f"Numeric field for EDA: {numeric_field_id}")
        # Remove outliers and filter where values > threshold
        threshold = np.nanmean(df[numeric_field_id])
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize the numeric column
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to group by a likely categorical field
        candidate_categorical = None
        for col in df.columns:
            if col != numeric_field_id and df[col].nunique() < min(10, len(df) // 10):
                candidate_categorical = col
                break
        if candidate_categorical:
            group_field_id = candidate_categorical
            print(f"\nGrouped by: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print("No numeric field found for EDA; please explore columns above and select one analytically.")
else:
    print("No dataframes extracted. Please check prior steps.")

## 5. Visualization
Visualize the distribution of the selected numeric field and relationship to a categorical group (if found).

Below, an example histogram and, if available, boxplot by group is displayed.

In [ ]:
import matplotlib.pyplot as plt

if dataframes and 'numeric_field_id' in locals():
    fig, ax = plt.subplots(figsize=(7,4))
    filtered_df[numeric_field_id].hist(bins=20, ax=ax, alpha=0.7)
    ax.set_title(f"Distribution of {numeric_field_id}")
    ax.set_xlabel(numeric_field_id)
    ax.set_ylabel("Frequency")
    plt.show()

    # Boxplot by categorical group if exists
    if 'group_field_id' in locals() and group_field_id in filtered_df.columns:
        plt.figure(figsize=(10,4))
        filtered_df.boxplot(column=numeric_field_id, by=group_field_id)
        plt.title(f"Boxplot of {numeric_field_id} by {group_field_id}")
        plt.suptitle('')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No visualization rendered due to lack of data or field selection.")

## 6. Conclusion
This notebook demonstrated how to load, explore, and process the FAIR² "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" dataset using the `mlcroissant` library, referencing all entities by their Croissant `@id`. You can further extend this workflow for modeling, deeper analyses, or domain-specific interpretation.